# Myna Sleep Paralysis Survey: Tables, Figures, and Results Summary Notebook

This Google Colab notebook walks you step-by-step through creating the Week 7 deliverables:

- **Table 1:** Sample characteristics  
- **Table 2:** Sleep paralysis frequency/severity summary  
- **Table 3:** Regression model results  
- **Figure 2:** Random Forest feature importance plot  
- **Figure 3:** Overall findings/results model  

You will create **Figure 1: Conceptual model of study design** separately in PowerPoint, Canva, BioRender, Lucidchart, or another drawing tool.

## Important instructions
1. Run the notebook from top to bottom.
2. Read every blue/markdown instruction before running the code below it.
3. When you see `TODO`, update the variable names to match your Google Form column names.
4. If you get an error, first ask Gemini/Colab AI to explain it.
5. If Gemini cannot fix it, ping Dr. Gamble-George with:
   - the section title,
   - a screenshot of the error,
   - the code cell that failed,
   - and the column name/question involved.

## Data privacy
Only upload the **anonymous survey export**. Do not upload emails, names, IP addresses, or any identifiable/private information.


# Part 0: Load packages

Run this first. It loads the Python tools needed for tables, figures, regression, and Random Forest.


In [ ]:
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt

from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import (
    r2_score, mean_absolute_error,
    accuracy_score, classification_report, confusion_matrix
)

print("Packages loaded successfully.")

# Part 1: Upload your survey data

Export your Google Form responses as a **CSV** file if possible.

In Google Forms:
1. Go to **Responses**
2. Click the green Sheets icon
3. In Google Sheets, go to **File > Download > Comma-separated values (.csv)**
4. Upload that CSV file when prompted below


In [ ]:
from google.colab import files

uploaded = files.upload()
file_name = list(uploaded.keys())[0]

if file_name.lower().endswith(".csv"):
    df_raw = pd.read_csv(file_name)
elif file_name.lower().endswith((".xlsx", ".xls")):
    df_raw = pd.read_excel(file_name)
else:
    raise ValueError("Please upload a .csv, .xlsx, or .xls file.")

print("Data uploaded successfully.")
print("Rows, columns:", df_raw.shape)
df_raw.head()

# Part 2: Inspect your survey columns

Run this cell and look carefully at the column list. You will use these column names to fill in the TODO sections below.


In [ ]:
for i, col in enumerate(df_raw.columns):
    print(f"{i}: {col}")

# Part 3: Clean column names

This creates shorter Python-friendly column names. The notebook will print the original column and the cleaned column side by side.

Use the **cleaned column names** in the TODO sections.


In [ ]:
def clean_col_name(col):
    col = str(col).strip().lower()
    col = re.sub(r"[^a-z0-9]+", "_", col)
    col = re.sub(r"_+", "_", col).strip("_")
    return col[:90]

df = df_raw.copy()
original_to_clean = pd.DataFrame({
    "original_column": df_raw.columns,
    "clean_column": [clean_col_name(c) for c in df_raw.columns]
})
df.columns = original_to_clean["clean_column"]

display(original_to_clean)
df.head()

# Part 4: Create a codebook starter

This table helps you label each variable as a predictor, outcome, covariate, or screening variable.

You can download this and use it while writing the Methods section.


In [ ]:
codebook = pd.DataFrame({
    "clean_column": df.columns,
    "original_column": df_raw.columns,
    "role": "",
    "notes": ""
})

display(codebook)

codebook.to_csv("codebook_starter.csv", index=False)

# Part 5: Basic data quality checks

This checks:
- number of responses,
- missing data,
- exact duplicate rows.

This will help you write the Methods and Limitations sections.


In [ ]:
print("Total responses:", len(df))
print("Number of survey columns:", df.shape[1])

print("\nMissing values by column:")
missing = df.isna().sum().sort_values(ascending=False)
display(missing.to_frame("missing_count"))

print("\nExact duplicate rows:", df.duplicated().sum())

# Part 6: Define eligibility columns and create the final analytic dataset

Update the TODO column names below after looking at your cleaned column list.

The goal is to keep only participants who:
- are 18 or older,
- have experienced sleep paralysis,
- and consented, if your dataset includes a consent column.

If the survey already screened people out, this section may not remove many rows.


In [ ]:
# ============================================================
# ELIGIBILITY SCREENING — INITIAL SCREEN
# ============================================================

CONSENT_COL = None

AGE_18_COL = (
    "you_must_be_over_the_age_of_18_to_take_this_survey_"
    "please_confirm_that_you_are_18_or_above"
)

EVER_SP_COL = (
    "have_you_ever_experienced_sleep_paralysis"
)

df_clean = df.copy()


def is_yes(series):
    return (
        series.astype(str)
        .str.strip()
        .str.lower()
        .eq("yes")
    )


start_n = len(df_clean)

# Consent, if present
if CONSENT_COL and CONSENT_COL in df_clean.columns:
    df_clean = df_clean[
        is_yes(df_clean[CONSENT_COL])
    ]

# Age 18+
df_clean = df_clean[
    is_yes(df_clean[AGE_18_COL])
]

# Must have experienced sleep paralysis
df_clean = df_clean[
    is_yes(df_clean[EVER_SP_COL])
]


print("Starting responses:", start_n)
print("Sample after age/SP screening:", len(df_clean))
print("Excluded at initial screening:", start_n - len(df_clean))

df_clean.head()

# Part 7: Convert survey answers to numbers

This converts common 1-5 survey responses into numeric values.

If your Google Form already uses numbers, this will still work.

If your wording is different, update the `likert_map` dictionary.


In [ ]:
### requires simplified column names
df_clean.loc[5, "do_you_think_sleep_paralysis_is_physiological_or_supernatural"] = "Entirely physiological"
df_clean.loc[47, "do_you_think_sleep_paralysis_is_physiological_or_supernatural"] = "Entirely physiological"
df_clean.loc[124, "do_you_think_sleep_paralysis_is_physiological_or_supernatural"] = "Mostly physiological"
df_clean.loc[164, "do_you_think_sleep_paralysis_is_physiological_or_supernatural"] = "Entirely physiological"
df_clean.loc[164, "do_you_think_sleep_paralysis_is_physiological_or_supernatural"] = "Entirely physiological"
df_clean.loc[22, "how_many_sleep_paralysis_episodes_have_you_experienced_in_your_lifetime_if_unsure_your_bes_num"] = "More than 100"
df_clean.loc[43, "how_many_sleep_paralysis_episodes_have_you_experienced_in_your_lifetime_if_unsure_your_bes_num"] = "6-10"
df_clean.loc[47, "how_many_sleep_paralysis_episodes_have_you_experienced_in_your_lifetime_if_unsure_your_bes_num"] = "More than 100"
df_clean.loc[78, "how_many_sleep_paralysis_episodes_have_you_experienced_in_your_lifetime_if_unsure_your_bes_num"] = "More than 100"
df_clean.loc[78, "how_many_sleep_paralysis_episodes_have_you_experienced_in_your_lifetime_if_unsure_your_bes_num"] = "More than 100"
df_clean.loc[103, "how_many_sleep_paralysis_episodes_have_you_experienced_in_your_lifetime_if_unsure_your_bes_num"] = "More than 100"
df_clean.loc[151, "how_many_sleep_paralysis_episodes_have_you_experienced_in_your_lifetime_if_unsure_your_bes_num"] = "More than 100"
df_clean.loc[172, "how_many_sleep_paralysis_episodes_have_you_experienced_in_your_lifetime_if_unsure_your_bes_num"] = "11-20"
df_clean.loc[173, "how_many_sleep_paralysis_episodes_have_you_experienced_in_your_lifetime_if_unsure_your_bes_num"] = "More than 100"
df_clean.loc[186, "how_many_sleep_paralysis_episodes_have_you_experienced_in_your_lifetime_if_unsure_your_bes_num"] = "More than 100"

conditions = {
    "depression": "Depression",
    "anxiety": "Anxiety disorder",
    "ptsd": "PTSD",
    "panic_disorder": "Panic disorder"
}

for variable, option in conditions.items():
    df_clean[variable] = (
        df_clean["have_you_ever_been_diagnosed_by_a_healthcare_professional_with_any_of_the_following_select"]
        .fillna("")
        .str.contains(option, case=False, na=False)
        .astype(int)
    )

likert_map = {
    "no": 0,
    "n/a": 0,
    "nil": 0,
    "no strong preference.":0,
    "none":0,
    "nope":0,
    "nope!":0,
    "none of the above": 0,
    "prefer not to say": 0,
    "yes": 1,
    ##
    "never": 1,
    "rarely": 2,
    "sometimes": 3,
    "often": 4,
    "almost always": 5,
    ##
    "not at all": 1,
    "a little": 2,
    "somewhat": 3,
    "quite a bit": 4,
    "extensively": 5,
    ##
    "not at all": 1,
    "slightly": 2,
    "moderately": 3,
    "very": 4,
    "extremely": 5,
    ##
    "very negative": 1,
    "somewhat negative": 2,
    "neutral": 3,
    "somewhat positive": 4,
    "very positive": 5,
    ##
    "less than 5 minutes": 1,
    "5-30 minutes": 2,
    "several hours": 3,
    "the rest of the day": 4,
    "multiple days": 5,
    ##
    "entirely physiological": 1,
    "mostly physiological": 2,
    "both physiological and supernatural": 3,
    "mostly supernatural": 4,
    "entirely supernatural": 5,
    "spiritual": 5,
    ##
    "1": 1,
    "2-5": 2,
    "6-10": 3,
    "11-20": 4,
    "21-50": 5,
    "51-100": 6,
    "More than 100": 7
}

def convert_to_numeric(series):
    numeric = pd.to_numeric(series, errors="coerce")
    mapped = series.astype(str).str.strip().str.lower().map(likert_map)
    return numeric.fillna(mapped)

df_num = df_clean.copy()

for col in df_clean.columns:
    df_num[col + "_num"] = convert_to_numeric(df_clean[col])

print("Numeric columns added.")
print("Rows, columns:", df_num.shape)
df_num.head()

# Part 8: Set your main variables

This is the most important setup section.

Replace the example column names with your actual cleaned column names.

## Suggested roles

### Sample characteristics/covariates
Examples: age, gender, student status, recruitment source, sleep quality, napping, family history.

### Sleep paralysis frequency/severity variables
Examples: lifetime SP episodes, SP episodes in past 6 months, fear intensity, emotional effects, sleep avoidance, daily functioning.

### Main predictors
Examples: dream awareness, lucid dream frequency, dream control, prior SP knowledge, supernatural/cultural beliefs, recognition of SP.

### Main outcome
Choose one primary outcome first, such as fear intensity.


In [ ]:
# TODO: Replace these with cleaned column names from your dataset.
# Use original cleaned column names for categorical variables.
# Use columns ending in _num for numeric variables.

eligibility_vars = [
    "you_must_be_over_the_age_of_18_to_take_this_survey_please_confirm_that_you_are_18_or_above_num",
    "have_you_ever_experienced_sleep_paralysis_num",
]

additional_covariate_vars = [
    "narcolepsy",
    "sleep_disorder",
    "anxiety",
    "depression",
]

predictor_vars = [
    "while_dreaming_how_often_do_you_realize_that_you_are_in_a_dream_num",
    "when_dreaming_how_often_do_you_have_control_over_your_actions_or_the_dream_environment_num",
    "while_dreaming_how_often_do_you_notice_that_something_in_the_dream_doesn_t_make_sense_num",
    ##
    "which_explanation_best_matches_your_current_beliefs_about_sleep_paralysis_num",
    "how_extensively_have_you_learned_about_sleep_paralysis_through_scientific_spiritual_or_oth_num",
    ##
    "how_many_sleep_paralysis_episodes_have_you_experienced_in_your_lifetime_if_unsure_your_bes_num",
    "how_many_sleep_paralysis_episodes_have_you_experienced_in_the_past_6_months_if_unsure_your_num",
    ##
    "during_your_sleep_paralysis_episodes_how_often_do_you_recognize_that_you_are_experiencing__num",
    "when_you_recognize_that_you_are_experiencing_sleep_paralysis_how_confident_are_you_that_it_num",
    "overall_how_do_you_view_your_sleep_paralysis_experiences_num",
]

outcome_vars = {
    "during_your_most_recent_sleep_paralysis_episode_how_frightened_did_you_feel_num",
    "how_long_do_the_emotional_effects_of_an_episode_usually_last_num",
}

secondary_outcome_vars = [
    "has_sleep_paralysis_ever_made_you_avoid_sleeping_num",
    "has_sleep_paralysis_affected_your_daily_functioning_num",
    "have_you_sought_medical_or_psychological_help_because_of_sleep_paralysis",
]

# Short labels for the correlation figure
short_names = {
    "while_dreaming_how_often_do_you_realize_that_you_are_in_a_dream_num":
        "Dream awareness",

    "when_dreaming_how_often_do_you_have_control_over_your_actions_or_the_dream_environment_num":
        "Dream control",

    "while_dreaming_how_often_do_you_notice_that_something_in_the_dream_doesn_t_make_sense_num":
        "Dream skepticism",

    "which_explanation_best_matches_your_current_beliefs_about_sleep_paralysis_num":
        "Supernatural SP beliefs",

    "how_extensively_have_you_learned_about_sleep_paralysis_through_scientific_spiritual_or_oth_num":
        "SP knowledge",

    "how_many_sleep_paralysis_episodes_have_you_experienced_in_your_lifetime_if_unsure_your_bes_num":
        "Lifetime SP frequency",

    "how_many_sleep_paralysis_episodes_have_you_experienced_in_the_past_6_months_if_unsure_your_num":
        "6-month SP frequency",

    "during_your_sleep_paralysis_episodes_how_often_do_you_recognize_that_you_are_experiencing__num":
        "SP recognition",

    "when_you_recognize_that_you_are_experiencing_sleep_paralysis_how_confident_are_you_that_it_num":
        "Threat confidence",

    "overall_how_do_you_view_your_sleep_paralysis_experiences_num":
        "Positive SP outlook",

    "during_your_most_recent_sleep_paralysis_episode_how_frightened_did_you_feel_num":
        "Fear",

    "how_long_do_the_emotional_effects_of_an_episode_usually_last_num":
        "Emotional effects",

    "has_sleep_paralysis_ever_made_you_avoid_sleeping_num":
        "Sleep avoidance",

    "has_sleep_paralysis_affected_your_daily_functioning_num":
        "Impacted daily functioning",

    "have_you_sought_medical_or_psychological_help_because_of_sleep_paralysis_num":
        "Help-seeking"
}

print("Eligibility vars found:", [v for v in eligibility_vars if v in df_num.columns])
print("Predictor vars found:", [v for v in predictor_vars if v in df_num.columns])
print("Covariate vars found:", [v for v in additional_covariate_vars if v in df_num.columns])
print("Primary outcomes:", outcome_vars)
print("Secondary outcome vars found:", [v for v in secondary_outcome_vars if v in df_num.columns])

# Deliverable: Table 1 - Sample characteristics

This creates a manuscript-ready sample characteristics table.

Use this for:
- age,
- gender,
- recruitment platform/source,
- student status,
- other demographic variables if collected.

If you did not collect demographics, keep this table focused on recruitment/source and eligibility counts.


In [ ]:
recruitment_source_col = "where_did_you_find_out_about_this_survey"

age_col = "you_must_be_over_the_age_of_18_to_take_this_survey_please_confirm_that_you_are_18_or_above"
sp_col = "have_you_ever_experienced_sleep_paralysis"

table1_parts = []


# Recruitment
recruitment = (
    df_clean[recruitment_source_col]
    .fillna("Missing")
    .value_counts()
    .rename_axis("Category")
    .reset_index(name="n")
)

recruitment["Variable"] = "Recruitment source"
recruitment["%"] = (recruitment["n"] / len(df_clean) * 100).round(1)

table1_parts.append(
    recruitment[["Variable", "Category", "n", "%"]]
)


# Age eligibility
age = (
    df_num[age_col + "_num"]
    .map({1: "≥18 years", 0: "<18 years"})
    .fillna("Missing")
    .value_counts()
    .rename_axis("Category")
    .reset_index(name="n")
)

age["Variable"] = "Age eligibility"
age["%"] = (age["n"] / len(df_num) * 100).round(1)

table1_parts.append(
    age[["Variable", "Category", "n", "%"]]
)


# SP eligibility
sp = (
    df_num[sp_col + "_num"]
    .map({1: "Experienced SP", 0: "No SP experience"})
    .fillna("Missing")
    .value_counts()
    .rename_axis("Category")
    .reset_index(name="n")
)

sp["Variable"] = "SP eligibility"
sp["%"] = (sp["n"] / len(df_num) * 100).round(1)

table1_parts.append(
    sp[["Variable", "Category", "n", "%"]]
)

# ============================================================
# Narcolepsy exclusion
# ============================================================

narcolepsy_col = (
    "have_you_ever_been_diagnosed_by_a_healthcare_"
    "professional_with_narcolepsy"
)

# Rebuild df_num from the current 214-person df_clean
# so we know narcolepsy has NOT already been excluded.
df_num = df_clean.copy()

for col in df_clean.columns:
    df_num[col + "_num"] = convert_to_numeric(df_clean[col])

print(
    "Sample before narcolepsy exclusion:",
    len(df_num)
)

narcolepsy_yes = (
    df_num[narcolepsy_col]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("yes")
)

print(
    "Narcolepsy exclusions:",
    int(narcolepsy_yes.sum())
)

# Keep a copy for Table 1 counts
df_all = df_num.copy()

# Exclude participants diagnosed with narcolepsy
df_num = df_num.loc[
    ~narcolepsy_yes
].copy()

print(
    "Final analytic sample after narcolepsy exclusion:",
    len(df_num)
)

narcolepsy = (
    df_all[narcolepsy_col + "_num"]
    .map({
        1: "Narcolepsy: excluded",
        0: "No narcolepsy: retained"
    })
    .fillna("Missing")
    .value_counts()
    .rename_axis("Category")
    .reset_index(name="n")
)

narcolepsy["Variable"] = "Narcolepsy screening"
narcolepsy["%"] = (narcolepsy["n"] / len(df_all) * 100).round(1)

table1_parts.append(
    narcolepsy[["Variable", "Category", "n", "%"]]
)

# Final table
table1 = pd.concat(table1_parts, ignore_index=True)

display(table1)

# Deliverable: Table 2 - Sleep paralysis frequency and severity summary

This table summarizes the main sleep paralysis variables.

It can include:
- lifetime number of SP episodes,
- SP episodes in the past 6 months,
- fear intensity,
- emotional effects,
- sleep avoidance,
- daily functioning,
- positive/negative interpretation.


In [ ]:
descriptive_vars = [
    #frequency
    "how_many_sleep_paralysis_episodes_have_you_experienced_in_your_lifetime_if_unsure_your_bes_num",
    "how_many_sleep_paralysis_episodes_have_you_experienced_in_the_past_6_months_if_unsure_your_num",
    #experience
    "during_your_most_recent_sleep_paralysis_episode_how_frightened_did_you_feel_num",
    "how_long_do_the_emotional_effects_of_an_episode_usually_last_num",
    "has_sleep_paralysis_ever_made_you_avoid_sleeping_num",
    "has_sleep_paralysis_affected_your_daily_functioning_num",
    #interpretation
    "overall_how_do_you_view_your_sleep_paralysis_experiences_num",
]

# Figure: Distributions of key sleep paralysis variables

This creates separate histograms for the variables in Table 2. These are helpful for checking whether variables are skewed.


In [ ]:
if len(descriptive_vars) == 0:

    print("Add variables to descriptive_vars first.")

else:

    for col in descriptive_vars:

        short_name = short_names.get(col, col)

        plt.figure(figsize=(7, 4))

        data = df_num[col].dropna()

        min_val = int(data.min())
        max_val = int(data.max())

        data.hist(
        bins=np.arange(min_val - 0.5, max_val + 1.5, 1),
        rwidth=0.4
        )

        plt.xticks(range(min_val, max_val + 1))

        plt.title(f"Distribution of {short_name}")

        plt.xlabel(short_name)

        plt.ylabel("Number of participants")

        plt.grid(axis="y", alpha=0.3)
        plt.gca().set_axisbelow(True)

        plt.tight_layout()

        filename = f"distribution_{short_name[:40]}.png".replace("/", "_")

        plt.savefig(filename, dpi=300, bbox_inches="tight")

        plt.show()

        files.download(filename)

# Correlation table and correlation figure

This helps you see which predictors are associated with the outcome.

Use this as preliminary evidence before the regression and Random Forest.


In [ ]:
analysis_vars = list(dict.fromkeys(
    predictor_vars + descriptive_vars
))

if len(analysis_vars) < 2:
    print("Add at least two numeric analysis variables.")
else:
    corr = df_num[analysis_vars].corr(method="spearman")

    corr_table = corr.rename(
    index=short_names,
    columns=short_names
)

display(corr_table.round(2))

analysis_vars = list(dict.fromkeys(
    v + "_num"
    for v in predictor_vars + descriptive_vars
    if v + "_num" in df_num.columns
))

if len(analysis_vars) < 2:
    print("Add at least two numeric analysis variables.")
else:
    corr = df_num[analysis_vars].corr(method="spearman")

plot_labels = [short_names.get(col, col) for col in corr.columns]

plt.figure(figsize=(12, 10))

plt.imshow(
    corr,
    aspect="auto",
    vmin=-1,
    vmax=1
)

plt.colorbar(label="Spearman correlation")

plt.xticks(
    range(len(corr.columns)),
    plot_labels,
    rotation=45,
    ha="right"
)

plt.yticks(
    range(len(corr.index)),
    plot_labels
)

plt.title("Spearman Correlation Matrix")
plt.tight_layout()

plt.savefig(
    "Figure_correlation_matrix.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()
files.download("Figure_correlation_matrix.png")

# Deliverable: Table 3 - Regression model results

This runs an exploratory regression model predicting your primary outcome.

## Choose the correct model
- If your outcome is numeric, such as fear intensity from 1 to 5, use **linear regression**.
- If your outcome is binary, such as high fear vs low fear, use **logistic regression**.

This first table is a simple exploratory linear regression output.


In [ ]:
# ============================================================
# Table 3: Exploratory linear regression models
# ============================================================

existing_predictors = [
    v for v in predictor_vars
    if v in df_num.columns
]

if len(existing_predictors) == 0:

    print("Add predictor variables in Part 8 first.")

else:

    all_summaries = []
    all_results = {}

    for outcome_col in outcome_vars:

        outcome_label = short_names.get(outcome_col, outcome_col)

        print("\n" + "="*70)
        print(f"Outcome: {outcome_label}")
        print("="*70)

        if outcome_col not in df_num.columns:
            print(f"Outcome not found: {outcome_col}")
            continue

        # ----------------------------------------------------
        # Create model dataset
        # ----------------------------------------------------

        model_df = df_num[
            [outcome_col] + existing_predictors
        ].copy()

        model_df = model_df.dropna(
            subset=[outcome_col]
        )

        X = model_df[existing_predictors]
        y = model_df[outcome_col]

        # ----------------------------------------------------
        # Identify numeric and categorical predictors
        # ----------------------------------------------------

        numeric_features = X.select_dtypes(
            include=[np.number]
        ).columns.tolist()

        categorical_features = [
            c for c in X.columns
            if c not in numeric_features
        ]

        # ----------------------------------------------------
        # Preprocessing
        # ----------------------------------------------------

        preprocessor = ColumnTransformer(
            transformers=[
                (
                    "num",
                    Pipeline([
                        ("imputer", SimpleImputer(strategy="median")),
                        ("scaler", StandardScaler())
                    ]),
                    numeric_features
                ),
                (
                    "cat",
                    Pipeline([
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        ("onehot", OneHotEncoder(handle_unknown="ignore"))
                    ]),
                    categorical_features
                )
            ]
        )

        # ----------------------------------------------------
        # Linear regression
        # ----------------------------------------------------

        lin_model = Pipeline([
            ("preprocess", preprocessor),
            ("model", LinearRegression())
        ])

        lin_model.fit(X, y)

        preds = lin_model.predict(X)

        # ----------------------------------------------------
        # Feature names
        # ----------------------------------------------------

        feature_names = []

        if numeric_features:
            feature_names.extend(numeric_features)

        if categorical_features:

            ohe = (
                lin_model
                .named_steps["preprocess"]
                .named_transformers["cat"]
                .named_steps["onehot"]
            )

            feature_names.extend(
                ohe.get_feature_names_out(
                    categorical_features
                ).tolist()
            )

        # ----------------------------------------------------
        # Coefficients
        # ----------------------------------------------------

        coefs = lin_model.named_steps["model"].coef_

        table = pd.DataFrame({
            "Predictor": [
                short_names.get(x, x)
                for x in feature_names
            ],
            "Coefficient": coefs
        }).sort_values(
            "Coefficient",
            key=abs,
            ascending=False
        )

        all_results[outcome_label] = table

        # ----------------------------------------------------
        # Model summary
        # ----------------------------------------------------

        summary = pd.DataFrame({
            "Outcome": [outcome_label],
            "N": [len(model_df)],
            "R-squared": [
                round(r2_score(y, preds), 3)
            ],
            "Mean absolute error": [
                round(mean_absolute_error(y, preds), 3)
            ]
        })

        all_summaries.append(summary)

        # ----------------------------------------------------
        # Display
        # ----------------------------------------------------

        print(f"\nModel summary: {outcome_label}")
        display(summary)

        print(f"Regression results: {outcome_label}")
        display(table.round(3))

    # --------------------------------------------------------
    # Combined model summary
    # --------------------------------------------------------

    model_summary = pd.concat(
        all_summaries,
        ignore_index=True
    )

    print("\nModel summaries")
    display(model_summary)

    # --------------------------------------------------------
    # Save summary
    # --------------------------------------------------------

    model_summary.to_csv(
        "Table3_regression_model_summaries.csv",
        index=False
    )

    files.download(
        "Table3_regression_model_summaries.csv"
    )

    # --------------------------------------------------------
    # Save individual regression tables
    # --------------------------------------------------------

    for outcome_label, table in all_results.items():

        filename = (
            "Table3_"
            + outcome_label.lower().replace(" ", "_")
            + "_regression.csv"
        )

        table.to_csv(
            filename,
            index=False
        )

        files.download(filename)

# Optional: Binary logistic regression

Use this if you want to predict a binary outcome, such as **high fear** vs **low/moderate fear**.

For example, if fear is rated 1-5:
- 0 = fear score below 4
- 1 = fear score of 4 or 5


In [ ]:
# ============================================================
# Table 4: Exploratory binary logistic regression models
# ============================================================

existing_predictors = [
    v for v in predictor_vars
    if v in df_num.columns
]

if len(existing_predictors) == 0:

    print("Add predictor variables in Part 8 first.")

else:

    all_log_summaries = []
    all_log_results = {}

    for outcome_col in secondary_outcome_vars:

        outcome_label = short_names.get(outcome_col, outcome_col)

        print("\n" + "="*70)
        print(f"Outcome: {outcome_label}")
        print("="*70)

        if outcome_col not in df_num.columns:
            print(f"Outcome not found: {outcome_col}")
            continue

        # ----------------------------------------------------
        # Create model dataset
        # ----------------------------------------------------

        model_df = df_num[
            [outcome_col] + existing_predictors
        ].copy()

        model_df = model_df.dropna(
            subset=[outcome_col]
        )

        X = model_df[existing_predictors]
        y = model_df[outcome_col]

        # ----------------------------------------------------
        # Check that outcome is binary
        # ----------------------------------------------------

        if y.nunique() < 2:
            print("There is only one outcome class. Logistic regression cannot run.")
            continue

        # ----------------------------------------------------
        # Identify numeric and categorical predictors
        # ----------------------------------------------------

        numeric_features = X.select_dtypes(
            include=[np.number]
        ).columns.tolist()

        categorical_features = [
            c for c in X.columns
            if c not in numeric_features
        ]

        # ----------------------------------------------------
        # Preprocessing
        # ----------------------------------------------------

        preprocessor = ColumnTransformer(
            transformers=[
                (
                    "num",
                    Pipeline([
                        ("imputer", SimpleImputer(strategy="median")),
                        ("scaler", StandardScaler())
                    ]),
                    numeric_features
                ),
                (
                    "cat",
                    Pipeline([
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        ("onehot", OneHotEncoder(handle_unknown="ignore"))
                    ]),
                    categorical_features
                )
            ]
        )

        # ----------------------------------------------------
        # Logistic regression
        # ----------------------------------------------------

        log_model = Pipeline([
            ("preprocess", preprocessor),
            ("model", LogisticRegression(max_iter=1000))
        ])

        log_model.fit(X, y)

        preds = log_model.predict(X)

        # ----------------------------------------------------
        # Feature names
        # ----------------------------------------------------

        feature_names = []

        if numeric_features:
            feature_names.extend(numeric_features)

        if categorical_features:

            ohe = (
                log_model
                .named_steps["preprocess"]
                .named_transformers["cat"]
                .named_steps["onehot"]
            )

            feature_names.extend(
                ohe.get_feature_names_out(
                    categorical_features
                ).tolist()
            )

        # ----------------------------------------------------
        # Coefficients and odds ratios
        # ----------------------------------------------------

        coefs = log_model.named_steps["model"].coef_[0]

        odds_ratios = np.exp(coefs)

        table = pd.DataFrame({
            "Predictor": [
                short_names.get(x, x)
                for x in feature_names
            ],
            "Coefficient": coefs,
            "Odds ratio": odds_ratios
        }).sort_values(
            "Odds ratio",
            ascending=False
        )

        all_log_results[outcome_label] = table

        # ----------------------------------------------------
        # Model summary
        # ----------------------------------------------------

        summary = pd.DataFrame({
            "Outcome": [outcome_label],
            "N": [len(model_df)],
            "Accuracy": [
                round(accuracy_score(y, preds), 3)
            ]
        })

        all_log_summaries.append(summary)

        # ----------------------------------------------------
        # Display
        # ----------------------------------------------------

        print(f"\nModel summary: {outcome_label}")
        display(summary)

        print(f"Logistic regression results: {outcome_label}")
        display(table.round(3))

        print("\nClassification report:")
        print(classification_report(y, preds))

    # --------------------------------------------------------
    # Combined model summary
    # --------------------------------------------------------

    if len(all_log_summaries) > 0:

        logistic_summary = pd.concat(
            all_log_summaries,
            ignore_index=True
        )

        print("\nLogistic regression model summaries")
        display(logistic_summary)

        # ----------------------------------------------------
        # Save summary
        # ----------------------------------------------------

        logistic_summary.to_csv(
            "Table4_logistic_regression_model_summaries.csv",
            index=False
        )

        files.download(
            "Table4_logistic_regression_model_summaries.csv"
        )

        # ----------------------------------------------------
        # Save individual regression tables
        # ----------------------------------------------------

        for outcome_label, table in all_log_results.items():

            filename = (
                "Table4_"
                + outcome_label.lower().replace(" ", "_")
                + "_logistic_regression.csv"
            )

            table.to_csv(
                filename,
                index=False
            )

            files.download(filename)

# Deliverable: Figure 2 - Random Forest feature importance plot

This model identifies which variables are most useful for predicting the primary outcome.

## Important interpretation note
Random Forest feature importance does **not** prove causation. It only shows which variables helped the model make predictions in this dataset.


In [ ]:
existing_predictors = [
    v for v in predictor_vars
    if v in df_num.columns
]

if len(existing_predictors) == 0:

    print("Add predictor variables first.")

else:

    all_importances = {}

    for outcome_col in outcome_vars:

        outcome_label = short_names.get(outcome_col, outcome_col)

        print("\n" + "="*70)
        print(f"Outcome: {outcome_label}")
        print("="*70)

        if outcome_col not in df_num.columns:
            print(f"Outcome not found: {outcome_col}")
            continue

        # ----------------------------------------------------
        # Create model dataset
        # ----------------------------------------------------

        rf_df = df_num[
            [outcome_col] + existing_predictors
        ].copy()

        rf_df = rf_df.dropna(
            subset=[outcome_col]
        )

        X = rf_df[existing_predictors]
        y = rf_df[outcome_col]

        # ----------------------------------------------------
        # Identify numeric and categorical predictors
        # ----------------------------------------------------

        numeric_features = X.select_dtypes(
            include=[np.number]
        ).columns.tolist()

        categorical_features = [
            c for c in X.columns
            if c not in numeric_features
        ]

        # ----------------------------------------------------
        # Preprocessing
        # ----------------------------------------------------

        preprocessor = ColumnTransformer(
            transformers=[
                (
                    "num",
                    SimpleImputer(strategy="median"),
                    numeric_features
                ),
                (
                    "cat",
                    Pipeline([
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        ("onehot", OneHotEncoder(handle_unknown="ignore"))
                    ]),
                    categorical_features
                )
            ]
        )

        # ----------------------------------------------------
        # Random Forest
        # ----------------------------------------------------

        rf_model = Pipeline([
            ("preprocess", preprocessor),
            ("model", RandomForestRegressor(
                n_estimators=500,
                random_state=42
            ))
        ])

        rf_model.fit(X, y)

        preds = rf_model.predict(X)

        # ----------------------------------------------------
        # Performance
        # ----------------------------------------------------

        print("N:", len(rf_df))
        print("R-squared:", round(r2_score(y, preds), 3))
        print(
            "Mean absolute error:",
            round(mean_absolute_error(y, preds), 3)
        )

        # ----------------------------------------------------
        # Feature names
        # ----------------------------------------------------

        feature_names = []

        if numeric_features:
            feature_names.extend(numeric_features)

        if categorical_features:

            ohe = (
                rf_model
                .named_steps["preprocess"]
                .named_transformers["cat"]
                .named_steps["onehot"]
            )

            feature_names.extend(
                ohe.get_feature_names_out(
                    categorical_features
                ).tolist()
            )

        # ----------------------------------------------------
        # Feature importance
        # ----------------------------------------------------

        importances = pd.DataFrame({
            "Feature": feature_names,
            "Importance": (
                rf_model
                .named_steps["model"]
                .feature_importances_
            )
        }).sort_values(
            "Importance",
            ascending=False
        )

        all_importances[outcome_label] = importances

        display(importances.head(20))

        # ----------------------------------------------------
        # Save feature importance table
        # ----------------------------------------------------

        filename = (
            "Random_forest_"
            + outcome_label.lower().replace(" ", "_")
            + "_feature_importance.csv"
        )

        importances.to_csv(
            filename,
            index=False
        )

        files.download(filename)

        # ----------------------------------------------------
        # Plot
        # ----------------------------------------------------

        top = (
            importances
            .head(10)
            .sort_values("Importance")
        )

        plt.figure(figsize=(9, 6))

        plt.barh(
            [
                short_names.get(x, x)
                for x in top["Feature"]
            ],
            top["Importance"]
        )

        plt.xlabel("Feature importance")
        plt.ylabel("Predictor")
        plt.title(
            f"Random Forest Feature Importance: {outcome_label}"
        )

        plt.tight_layout()

        figure_filename = (
            "Figure2_random_forest_"
            + outcome_label.lower().replace(" ", "_")
            + ".png"
        )

        plt.savefig(
            figure_filename,
            dpi=300,
            bbox_inches="tight"
        )

        plt.show()

        files.download(figure_filename)

# Deliverable: Figure 3 - Overall findings/results model

This creates a simple text-based findings model based on the top Random Forest predictors.

You can use this figure as a draft, then recreate a cleaner version in PowerPoint, Canva, or BioRender if needed.


In [ ]:
# ============================================================
# Exploratory regression models for multiple outcomes
# ============================================================

outcome_vars = {
    "Fear":
        "during_your_most_recent_sleep_paralysis_episode_how_frightened_did_you_feel_num",

    "Emotional effects":
        "how_long_do_the_emotional_effects_of_an_episode_usually_last_num",

    "Sleep avoidance":
        "has_sleep_paralysis_ever_made_you_avoid_sleeping_num",

    "Daily functioning":
        "has_sleep_paralysis_affected_your_daily_functioning_num"
}

# Use the same predictor set for every outcome
existing_predictors = [
    v for v in predictor_vars
    if v in df_num.columns
]

if len(existing_predictors) == 0:
    print("No predictor variables were found.")
else:

    all_model_summaries = []
    all_model_results = {}

    for outcome_label, outcome_col in outcome_vars.items():

        print("\n" + "="*70)
        print(f"OUTCOME: {outcome_label}")
        print("="*70)

        if outcome_col not in df_num.columns:
            print(f"Outcome column not found: {outcome_col}")
            continue

        # ----------------------------------------------------
        # Create analysis dataset
        # ----------------------------------------------------

        model_df = df_num[
            [outcome_col] + existing_predictors
        ].copy()

        # Remove rows missing the outcome
        model_df = model_df.dropna(subset=[outcome_col])

        X = model_df[existing_predictors]
        y = model_df[outcome_col]

        # ----------------------------------------------------
        # Identify numeric/categorical predictors
        # ----------------------------------------------------

        numeric_features = X.select_dtypes(
            include=[np.number]
        ).columns.tolist()

        categorical_features = [
            c for c in X.columns
            if c not in numeric_features
        ]

        # ----------------------------------------------------
        # Preprocessing
        # ----------------------------------------------------

        preprocessor = ColumnTransformer(
            transformers=[
                (
                    "num",
                    Pipeline([
                        ("imputer", SimpleImputer(strategy="median")),
                        ("scaler", StandardScaler())
                    ]),
                    numeric_features
                ),

                (
                    "cat",
                    Pipeline([
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        ("onehot", OneHotEncoder(handle_unknown="ignore"))
                    ]),
                    categorical_features
                )
            ]
        )

        # ----------------------------------------------------
        # Regression model
        # ----------------------------------------------------

        lin_model = Pipeline([
            ("preprocess", preprocessor),
            ("model", LinearRegression())
        ])

        lin_model.fit(X, y)

        # In-sample predictions
        preds = lin_model.predict(X)

        # ----------------------------------------------------
        # Get feature names
        # ----------------------------------------------------

        feature_names = []

        if numeric_features:
            feature_names.extend(numeric_features)

        if categorical_features:
            ohe = (
                lin_model
                .named_steps["preprocess"]
                .named_transformers["cat"]
                .named_steps["onehot"]
            )

            feature_names.extend(
                ohe.get_feature_names_out(
                    categorical_features
                ).tolist()
            )

        coefs = lin_model.named_steps["model"].coef_

        # ----------------------------------------------------
        # Regression results table
        # ----------------------------------------------------

        table = pd.DataFrame({
            "Predictor": [
                short_names.get(x, x)
                for x in feature_names
            ],
            "Coefficient": coefs
        })

        table = table.sort_values(
            "Coefficient",
            key=abs,
            ascending=False
        )

        all_model_results[outcome_label] = table

        # ----------------------------------------------------
        # Model summary
        # ----------------------------------------------------

        summary = pd.DataFrame({
            "Outcome": [outcome_label],
            "N": [len(model_df)],
            "R-squared": [
                round(r2_score(y, preds), 3)
            ],
            "MAE": [
                round(mean_absolute_error(y, preds), 3)
            ]
        })

        all_model_summaries.append(summary)

        # ----------------------------------------------------
        # Display
        # ----------------------------------------------------

        print(f"\nModel summary: {outcome_label}")
        display(summary)

        print(f"Regression results: {outcome_label}")
        display(table.round(3))


    # ========================================================
    # Combined model summary
    # ========================================================

    model_summary_all = pd.concat(
        all_model_summaries,
        ignore_index=True
    )

    print("\n" + "="*70)
    print("ALL MODEL SUMMARIES")
    print("="*70)

    display(model_summary_all)

    model_summary_all.to_csv(
        "Regression_model_summaries_all_outcomes.csv",
        index=False
    )

    files.download(
        "Regression_model_summaries_all_outcomes.csv"
    )


    # ========================================================
    # Save each outcome's regression table
    # ========================================================

    for outcome_label, table in all_model_results.items():

        safe_name = (
            outcome_label
            .lower()
            .replace(" ", "_")
        )

        table.to_csv(
            f"Regression_{safe_name}.csv",
            index=False
        )

        files.download(
            f"Regression_{safe_name}.csv"
        )

# Results summary paragraph generator

This creates draft sentence starters for the Results section.

Fill in the blanks or edit based on the actual output.


In [ ]:
print("Draft Results Sentences")
print("----------------------")
print(f"The final analytic sample included {len(df_num)} participants.")
print("Descriptive statistics were calculated for sleep paralysis frequency, fear intensity, distress, and functional impact.")
print("Spearman correlations were used to examine associations among dream awareness, prior knowledge, interpretation variables, and sleep paralysis outcomes.")
print("Exploratory regression models were used to examine predictors of the primary outcome.")
print("A Random Forest model was used to identify the strongest predictors based on feature importance.")
if "importances" in globals():
    print("\nTop Random Forest predictors:")
    for i, row in importances.head(5).iterrows():
        feature_label = short_names.get(
            row["Feature"],
            row["Feature"]
        )
        print(
            f"- {feature_label}: "
            f"{row['Importance']:.3f}"
        )
print("\nAll analyses were interpreted as exploratory associations rather than evidence of causation.")
print("\nInterpret results cautiously because the study used an anonymous, self-report, cross-sectional online convenience sample.")

# Manuscript checklist for Week 7

Before the next meeting, make sure you have:

- [ ] Table 1: Sample characteristics  
- [ ] Table 2: Sleep paralysis frequency/severity summary  
- [ ] Table 3: Regression model results  
- [ ] Figure 1: Conceptual model of study design, created outside Python  
- [ ] Figure 2: Random Forest feature importance plot  
- [ ] Figure 3: Overall findings/results model  
- [ ] 1 paragraph summarizing the main results  
- [ ] Notes on which findings are strongest and which are only exploratory  

## What to ping me about
Ping Dr. Gamble-George if:
- your data will not upload,
- a column name is confusing,
- a model gives an error,
- the Random Forest feature importance looks strange,
- or you are unsure which outcome should be the primary outcome.

Try Gemini/Colab AI first for basic coding errors, but ping me if the error affects the interpretation or analysis plan.


In [ ]:
# ============================================================
# MYNA SLEEP PARALYSIS STUDY — FINAL ANALYSIS CELL
# Paste this into ONE NEW CELL at the end of the notebook.
# It rebuilds the analytic dataset directly from df_raw.
# ============================================================

import os
import re
import warnings
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import spearmanr
from statsmodels.miscmodels.ordinal_model import OrderedModel
import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests

from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.model_selection import KFold, StratifiedKFold, cross_validate
from sklearn.inspection import permutation_importance

warnings.filterwarnings("ignore")


# ============================================================
# 0. LOAD DATA IF df_raw IS NOT ALREADY PRESENT
# ============================================================

try:
    df_raw
except NameError:
    from google.colab import files

    uploaded = files.upload()
    file_name = list(uploaded.keys())[0]

    if file_name.lower().endswith(".csv"):
        df_raw = pd.read_csv(file_name)
    elif file_name.lower().endswith((".xlsx", ".xls")):
        df_raw = pd.read_excel(file_name)
    else:
        raise ValueError("Please upload a CSV or Excel file.")


OUT = Path("/content/Myna_SP_Final_Analysis_Outputs")
OUT.mkdir(parents=True, exist_ok=True)

print("Raw data shape:", df_raw.shape)


# ============================================================
# 1. CLEAN COLUMN NAMES
# ============================================================

def clean_col_name(col):
    col = str(col).strip().lower()
    col = re.sub(r"[^a-z0-9]+", "_", col)
    col = re.sub(r"_+", "_", col).strip("_")
    return col[:90]


df = df_raw.copy()
df.columns = [clean_col_name(c) for c in df.columns]


# Exact cleaned survey columns
C = {
    "sp_ever":
        "have_you_ever_experienced_sleep_paralysis",

    "narcolepsy":
        "have_you_ever_been_diagnosed_by_a_healthcare_professional_with_narcolepsy",

    "other_sleep":
        "if_you_have_ever_been_diagnosed_with_any_other_sleep_disorder_please_specify",

    "dream_awareness":
        "while_dreaming_how_often_do_you_realize_that_you_are_in_a_dream",

    "dream_control":
        "when_dreaming_how_often_do_you_have_control_over_your_actions_or_the_dream_environment",

    "dream_skepticism":
        "while_dreaming_how_often_do_you_notice_that_something_in_the_dream_doesn_t_make_sense",

    "beliefs":
        "which_explanation_best_matches_your_current_beliefs_about_sleep_paralysis",

    "knowledge":
        "how_extensively_have_you_learned_about_sleep_paralysis_through_scientific_spiritual_or_oth",

    "lifetime_freq":
        "how_many_sleep_paralysis_episodes_have_you_experienced_in_your_lifetime_if_unsure_your_bes",

    "sixmo_freq":
        "how_many_sleep_paralysis_episodes_have_you_experienced_in_the_past_6_months_if_unsure_your",

    "sp_recognition":
        "during_your_sleep_paralysis_episodes_how_often_do_you_recognize_that_you_are_experiencing_",

    "threat_conf":
        "when_you_recognize_that_you_are_experiencing_sleep_paralysis_how_confident_are_you_that_it",

    "positive_outlook":
        "overall_how_do_you_view_your_sleep_paralysis_experiences",

    "fear":
        "during_your_most_recent_sleep_paralysis_episode_how_frightened_did_you_feel",

    "emotional":
        "how_long_do_the_emotional_effects_of_an_episode_usually_last",

    "avoidance":
        "has_sleep_paralysis_ever_made_you_avoid_sleeping",

    "functioning":
        "has_sleep_paralysis_affected_your_daily_functioning",

    "help":
        "have_you_sought_medical_or_psychological_help_because_of_sleep_paralysis",

    "diagnoses":
        "have_you_ever_been_diagnosed_by_a_healthcare_professional_with_any_of_the_following_select",

    "age18":
        "you_must_be_over_the_age_of_18_to_take_this_survey_please_confirm_that_you_are_18_or_above",

    "recruitment":
        "where_did_you_find_out_about_this_survey",
}


missing_cols = [v for v in C.values() if v not in df.columns]

if missing_cols:
    print("\nWARNING: Expected columns not found:")
    for x in missing_cols:
        print(" -", x)


# ============================================================
# 2. DEFINE ANALYTIC SAMPLE
# ============================================================

def lower_clean(series):
    return series.astype(str).str.strip().str.lower()


sp_yes = lower_clean(df[C["sp_ever"]]).eq("yes")
age_yes = lower_clean(df[C["age18"]]).eq("yes")
nar_no = lower_clean(df[C["narcolepsy"]]).eq("no")

analytic_mask = sp_yes & age_yes & nar_no

df_a = df.loc[analytic_mask].copy()


screening = pd.DataFrame({
    "Measure": [
        "Total responses collected",
        "Reported previous sleep paralysis",
        "Age 18 or older",
        "No professional narcolepsy diagnosis",
        "Final analytic sample"
    ],
    "n": [
        len(df),
        int(sp_yes.sum()),
        int(age_yes.sum()),
        int(nar_no.sum()),
        len(df_a)
    ]
})

screening["Percent_of_total"] = (
    screening["n"] / len(df) * 100
).round(1)


print("\nSAMPLE FLOW")
display(screening)

screening.to_csv(
    OUT / "Table_screening_sample_flow.csv",
    index=False
)

print(f"\nFINAL ANALYTIC SAMPLE: N = {len(df_a)}")


# ============================================================
# 3. RECODE VARIABLES
# ============================================================

freq_map = {
    "never": 0,
    "rarely": 1,
    "sometimes": 2,
    "often": 3,
    "almost always": 4
}


belief_map = {
    "entirely physiological": 1,
    "mostly physiological": 2,
    "both physiological and supernatural": 3,
    "mostly supernatural": 4,
    "entirely supernatural": 5
}


knowledge_map = {
    "not at all": 1,
    "a little": 2,
    "somewhat": 3,
    "quite a bit": 4,
    "extensively": 5
}


lifetime_map = {
    "1": 1,
    "2-5": 2,
    "6-10": 3,
    "11-20": 4,
    "21-50": 5,
    "51-100": 6,
    "more than 100": 7
}


sixmo_map = {
    "0": 0,
    "1": 1,
    "2-5": 2,
    "6-10": 3,
    "11-20": 4,
    "21-50": 5,
    "51-100": 6,
    "more than 100": 7
}


outlook_map = {
    "very negative": 1,
    "somewhat negative": 2,
    "neutral": 3,
    "somewhat positive": 4,
    "very positive": 5
}


fear_map = {
    "not at all": 1,
    "slightly": 2,
    "moderately": 3,
    "very": 4,
    "extremely": 5
}


emotional_map = {
    "less than 5 minutes": 1,
    "5-30 minutes": 2,
    "several hours": 3,
    "the rest of the day": 4,
    "multiple days": 5
}


yesno_map = {
    "no": 0,
    "yes": 1
}


def map_text(col, mapping):
    return lower_clean(df_a[col]).map(mapping)


analysis = pd.DataFrame(index=df_a.index)


# ----------------------------
# Predictors
# ----------------------------

analysis["Dream awareness"] = map_text(
    C["dream_awareness"], freq_map
)

analysis["Dream control"] = map_text(
    C["dream_control"], freq_map
)

analysis["Dream skepticism"] = map_text(
    C["dream_skepticism"], freq_map
)

analysis["Supernatural SP beliefs"] = map_text(
    C["beliefs"], belief_map
)

analysis["SP knowledge"] = map_text(
    C["knowledge"], knowledge_map
)

analysis["Lifetime SP frequency"] = map_text(
    C["lifetime_freq"], lifetime_map
)

analysis["6-month SP frequency"] = map_text(
    C["sixmo_freq"], sixmo_map
)

analysis["SP recognition"] = map_text(
    C["sp_recognition"], freq_map
)

analysis["Threat confidence"] = pd.to_numeric(
    df_a[C["threat_conf"]],
    errors="coerce"
)

analysis["Positive SP outlook"] = map_text(
    C["positive_outlook"], outlook_map
)


# ----------------------------
# Primary outcomes
# ----------------------------

analysis["Fear"] = map_text(
    C["fear"], fear_map
)

analysis["Emotional effects"] = map_text(
    C["emotional"], emotional_map
)


# ----------------------------
# Secondary outcomes
# ----------------------------

analysis["Sleep avoidance"] = map_text(
    C["avoidance"], yesno_map
)

analysis["Daily functioning"] = map_text(
    C["functioning"], yesno_map
)

analysis["Help-seeking"] = map_text(
    C["help"], yesno_map
)


# Recruitment source
analysis["Recruitment source"] = (
    df_a[C["recruitment"]]
    .astype(str)
    .str.strip()
)


# Optional descriptive mental-health diagnosis variable
dx = (
    df_a[C["diagnoses"]]
    .fillna("")
    .astype(str)
    .str.lower()
)

analysis[
    "Any anxiety/depression/PTSD/panic diagnosis"
] = (
    dx.str.contains(
        "anxiety|depression|ptsd|panic",
        regex=True
    )
).astype(int)


# ============================================================
# 4. IDENTIFY NONSTANDARD / UNCODED RESPONSES
# ============================================================

uncoded_rows = []


check_specs = [
    (
        "Supernatural SP beliefs",
        C["beliefs"],
        belief_map
    ),
    (
        "Lifetime SP frequency",
        C["lifetime_freq"],
        lifetime_map
    ),
    (
        "6-month SP frequency",
        C["sixmo_freq"],
        sixmo_map
    )
]


for label, raw_col, mapping in check_specs:

    raw_vals = (
        df_a[raw_col]
        .dropna()
        .astype(str)
        .str.strip()
    )

    normalized = raw_vals.str.lower()

    bad = raw_vals[
        ~normalized.isin(mapping.keys())
    ]

    for value, count in bad.value_counts().items():

        uncoded_rows.append({
            "Variable": label,
            "Uncoded response": value,
            "n": count
        })


uncoded = pd.DataFrame(uncoded_rows)


print("\nUNCODED NONSTANDARD RESPONSES")
display(uncoded)

uncoded.to_csv(
    OUT / "Uncoded_responses_for_review.csv",
    index=False
)


# ============================================================
# 5. FINAL CODEBOOK
# ============================================================

codebook = pd.DataFrame([
    [
        "Dream awareness",
        "Predictor",
        "0-4",
        "Higher = recognizes dreaming more often"
    ],
    [
        "Dream control",
        "Predictor",
        "0-4",
        "Higher = greater dream control"
    ],
    [
        "Dream skepticism",
        "Predictor",
        "0-4",
        "Higher = notices dream inconsistencies more often"
    ],
    [
        "Supernatural SP beliefs",
        "Predictor",
        "1-5",
        "Higher = more supernatural explanation"
    ],
    [
        "SP knowledge",
        "Predictor",
        "1-5",
        "Higher = greater prior knowledge"
    ],
    [
        "Lifetime SP frequency",
        "Predictor/covariate",
        "1-7 ordered category",
        "Higher = more lifetime SP episodes"
    ],
    [
        "6-month SP frequency",
        "Predictor/covariate",
        "0-7 ordered category",
        "Higher = more recent SP episodes"
    ],
    [
        "SP recognition",
        "Proximal predictor",
        "0-4",
        "Higher = recognizes SP more often"
    ],
    [
        "Threat confidence",
        "Proximal predictor",
        "1-5",
        "Higher = more confident perceived threat is not real"
    ],
    [
        "Positive SP outlook",
        "Proximal predictor",
        "1-5",
        "Higher = more positive view of SP"
    ],
    [
        "Fear",
        "Primary ordinal outcome",
        "1-5",
        "Higher = greater fear"
    ],
    [
        "Emotional effects",
        "Primary ordinal outcome",
        "1-5",
        "Higher = longer emotional effects"
    ],
    [
        "Sleep avoidance",
        "Secondary binary outcome",
        "0/1",
        "1 = yes"
    ],
    [
        "Daily functioning",
        "Secondary binary outcome",
        "0/1",
        "1 = yes"
    ],
    [
        "Help-seeking",
        "Secondary binary outcome",
        "0/1",
        "1 = yes"
    ],
],
columns=[
    "Variable",
    "Role",
    "Coding",
    "Direction"
])


codebook.to_csv(
    OUT / "Final_codebook.csv",
    index=False
)


# ============================================================
# 6. TABLE 1 — SAMPLE CHARACTERISTICS
# ============================================================

recruit = (
    analysis["Recruitment source"]
    .replace({"nan": "Missing"})
    .value_counts(dropna=False)
    .rename_axis("Category")
    .reset_index(name="n")
)

recruit["Percent"] = (
    recruit["n"] / len(analysis) * 100
).round(1)

recruit.insert(
    0,
    "Variable",
    "Recruitment source"
)


mental = (
    analysis[
        "Any anxiety/depression/PTSD/panic diagnosis"
    ]
    .map({
        0: "No listed mental-health diagnosis",
        1: "At least one listed mental-health diagnosis"
    })
    .value_counts()
    .rename_axis("Category")
    .reset_index(name="n")
)

mental["Percent"] = (
    mental["n"] / len(analysis) * 100
).round(1)

mental.insert(
    0,
    "Variable",
    "Mental-health diagnosis"
)


table1 = pd.concat(
    [recruit, mental],
    ignore_index=True
)


print("\nTABLE 1")
display(table1)

table1.to_csv(
    OUT / "Table1_sample_characteristics.csv",
    index=False
)


# ============================================================
# 7. TABLE 2 — DESCRIPTIVE STATISTICS
# ============================================================

main_numeric = [
    "Dream awareness",
    "Dream control",
    "Dream skepticism",
    "Supernatural SP beliefs",
    "SP knowledge",
    "Lifetime SP frequency",
    "6-month SP frequency",
    "SP recognition",
    "Threat confidence",
    "Positive SP outlook",
    "Fear",
    "Emotional effects"
]


desc = analysis[
    main_numeric
].describe().T


desc = desc.rename(columns={
    "count": "N",
    "mean": "Mean",
    "std": "SD",
    "min": "Min",
    "25%": "Q1",
    "50%": "Median",
    "75%": "Q3",
    "max": "Max"
})[
    [
        "N",
        "Mean",
        "SD",
        "Min",
        "Q1",
        "Median",
        "Q3",
        "Max"
    ]
].round(2)


binary_summary = []

for col in [
    "Sleep avoidance",
    "Daily functioning",
    "Help-seeking"
]:

    s = analysis[col].dropna()

    binary_summary.append({
        "Outcome": col,
        "N": len(s),
        "Yes_n": int((s == 1).sum()),
        "Yes_percent": round(
            (s == 1).mean() * 100,
            1
        )
    })


binary_summary = pd.DataFrame(
    binary_summary
)


print("\nTABLE 2 — ORDINAL VARIABLES")
display(desc)

print("\nTABLE 2 — BINARY OUTCOMES")
display(binary_summary)


desc.to_csv(
    OUT / "Table2_ordinal_descriptives.csv"
)

binary_summary.to_csv(
    OUT / "Table2_binary_outcomes.csv",
    index=False
)


# ============================================================
# 8. SPEARMAN CORRELATIONS + FDR CORRECTION
# ============================================================

corr_vars = [
    "Dream awareness",
    "Dream control",
    "Dream skepticism",
    "Supernatural SP beliefs",
    "SP knowledge",
    "Lifetime SP frequency",
    "6-month SP frequency",
    "SP recognition",
    "Threat confidence",
    "Positive SP outlook",
    "Fear",
    "Emotional effects",
    "Sleep avoidance",
    "Daily functioning",
    "Help-seeking"
]


pairs = []


for i in range(len(corr_vars)):

    for j in range(i + 1, len(corr_vars)):

        a = corr_vars[i]
        b = corr_vars[j]

        temp = analysis[
            [a, b]
        ].dropna()

        if len(temp) >= 3:

            rho, p = spearmanr(
                temp[a],
                temp[b]
            )

            pairs.append({
                "Variable_1": a,
                "Variable_2": b,
                "N": len(temp),
                "Spearman_rho": rho,
                "p_raw": p
            })


spearman_long = pd.DataFrame(
    pairs
)


spearman_long["q_FDR"] = multipletests(
    spearman_long["p_raw"],
    method="fdr_bh"
)[1]


spearman_long[
    "FDR_significant_q<0.05"
] = (
    spearman_long["q_FDR"] < 0.05
)


spearman_long = (
    spearman_long
    .sort_values("q_FDR")
)


spearman_long.to_csv(
    OUT / "Spearman_correlations_with_FDR.csv",
    index=False
)


print(
    "\nTOP SPEARMAN CORRELATIONS AFTER FDR"
)

display(
    spearman_long
    .head(30)
    .round(4)
)


# Correlation matrix
# ============================================================
# UPDATED FIGURE 2 — SPEARMAN CORRELATION MATRIX
# Larger labels + publication-ready formatting
# Uses VIRIDIS color scheme
# ============================================================

import matplotlib.pyplot as plt
import numpy as np

corr = analysis[corr_vars].corr(method="spearman")

corr.to_csv(
    OUT / "Spearman_correlation_matrix.csv"
)

# Larger figure for manuscript readability
fig, ax = plt.subplots(figsize=(15, 13))

im = ax.imshow(
    corr,
    cmap="viridis",
    vmin=-1,
    vmax=1,
    aspect="equal"
)

# Add Spearman rho values inside each cell
for i in range(len(corr.index)):
    for j in range(len(corr.columns)):

        value = corr.iloc[i, j]

        # Automatically choose white or black text
        # depending on background intensity
        text_color = (
            "white"
            if value < -0.35
            else "black"
        )

        ax.text(
            j,
            i,
            f"{value:.2f}",
            ha="center",
            va="center",
            fontsize=10,
            color=text_color
        )

# Colorbar
cbar = fig.colorbar(
    im,
    ax=ax,
    fraction=0.046,
    pad=0.04
)

cbar.set_label(
    "Spearman correlation",
    fontsize=18
)

cbar.ax.tick_params(
    labelsize=15
)

# X-axis labels
ax.set_xticks(
    np.arange(len(corr.columns))
)

ax.set_xticklabels(
    corr.columns,
    rotation=55,
    ha="right",
    fontsize=15
)

# Y-axis labels
ax.set_yticks(
    np.arange(len(corr.index))
)

ax.set_yticklabels(
    corr.index,
    fontsize=15
)

# Optional thin white grid between cells
ax.set_xticks(
    np.arange(-0.5, len(corr.columns), 1),
    minor=True
)

ax.set_yticks(
    np.arange(-0.5, len(corr.index), 1),
    minor=True
)

ax.grid(
    which="minor",
    linewidth=0.4,
    alpha=0.30
)

ax.tick_params(
    which="minor",
    bottom=False,
    left=False
)

# Do NOT place a title inside the figure for JEI.
# The figure title will go in the manuscript figure legend.

plt.tight_layout()

plt.savefig(
    OUT / "Figure2_Spearman_correlation_matrix.png",
    dpi=600,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# 9. ORDINAL LOGISTIC REGRESSION
#    PRIMARY OUTCOMES:
#    - FEAR
#    - EMOTIONAL EFFECTS
# ============================================================

predictors = [
    "Dream awareness",
    "Dream control",
    "Dream skepticism",
    "Supernatural SP beliefs",
    "SP knowledge",
    "Lifetime SP frequency",
    "6-month SP frequency",
    "SP recognition",
    "Threat confidence",
    "Positive SP outlook"
]


def ordinal_logit_table(
    data,
    outcome,
    predictors
):

    d = data[
        [outcome] + predictors
    ].dropna().copy()

    y = d[outcome].astype(int)

    X = d[
        predictors
    ].astype(float)

    model = OrderedModel(
        y,
        X,
        distr="logit"
    )

    fit = model.fit(
        method="bfgs",
        maxiter=2000,
        disp=False
    )

    ci = fit.conf_int().loc[
        predictors
    ]

    out = pd.DataFrame({
        "Outcome":
            outcome,

        "Predictor":
            predictors,

        "OR":
            np.exp(
                fit.params.loc[
                    predictors
                ].values
            ),

        "CI95_low":
            np.exp(
                ci[0].values
            ),

        "CI95_high":
            np.exp(
                ci[1].values
            ),

        "p_value":
            fit.pvalues.loc[
                predictors
            ].values,

        "N":
            len(d)
    })


    out[
        "Significant_p<0.05"
    ] = (
        out["p_value"] < 0.05
    )

    return fit, out


ordinal_tables = []
ordinal_fits = {}


for outcome in [
    "Fear",
    "Emotional effects"
]:

    fit, tab = ordinal_logit_table(
        analysis,
        outcome,
        predictors
    )

    ordinal_fits[
        outcome
    ] = fit

    ordinal_tables.append(
        tab
    )

    print(
        "\n" +
        "=" * 75
    )

    print(
        "ORDINAL LOGISTIC REGRESSION:",
        outcome
    )

    print(
        "N =",
        tab["N"].iloc[0]
    )

    display(
        tab
        .sort_values("p_value")
        .round(4)
    )


ordinal_results = pd.concat(
    ordinal_tables,
    ignore_index=True
)


ordinal_results.to_csv(
    OUT / "Table3_ordinal_logistic_regression.csv",
    index=False
)


# ============================================================
# 10. SECONDARY BINARY LOGISTIC REGRESSION
#
# One predictor at a time because help-seeking has relatively
# few events for a 10-predictor adjusted model.
#
# FDR correction is applied within each outcome.
# ============================================================

def univariable_binary_logit(
    data,
    outcome,
    predictors
):

    rows = []

    for pred in predictors:

        d = data[
            [outcome, pred]
        ].dropna().copy()

        y = d[
            outcome
        ].astype(int)

        X = sm.add_constant(
            d[
                [pred]
            ].astype(float),
            has_constant="add"
        )

        try:

            fit = sm.Logit(
                y,
                X
            ).fit(
                disp=False,
                maxiter=1000
            )

            beta = fit.params[
                pred
            ]

            lo, hi = (
                fit.conf_int()
                .loc[pred]
            )

            rows.append({
                "Outcome":
                    outcome,

                "Predictor":
                    pred,

                "N":
                    len(d),

                "Events":
                    int(y.sum()),

                "OR":
                    np.exp(beta),

                "CI95_low":
                    np.exp(lo),

                "CI95_high":
                    np.exp(hi),

                "p_value":
                    fit.pvalues[pred]
            })


        except Exception:

            rows.append({
                "Outcome":
                    outcome,

                "Predictor":
                    pred,

                "N":
                    len(d),

                "Events":
                    int(y.sum()),

                "OR":
                    np.nan,

                "CI95_low":
                    np.nan,

                "CI95_high":
                    np.nan,

                "p_value":
                    np.nan
            })


    tab = pd.DataFrame(
        rows
    )


    valid = tab[
        "p_value"
    ].notna()


    tab.loc[
        valid,
        "q_FDR"
    ] = multipletests(
        tab.loc[
            valid,
            "p_value"
        ],
        method="fdr_bh"
    )[1]


    tab[
        "FDR_significant_q<0.05"
    ] = (
        tab["q_FDR"] < 0.05
    )


    return tab


binary_tables = []


for outcome in [
    "Sleep avoidance",
    "Daily functioning",
    "Help-seeking"
]:

    tab = univariable_binary_logit(
        analysis,
        outcome,
        predictors
    )

    binary_tables.append(
        tab
    )

    print(
        "\n" +
        "=" * 75
    )

    print(
        "BINARY LOGISTIC REGRESSION:",
        outcome
    )

    print(
        "Outcome events =",
        int(
            analysis[
                outcome
            ].sum()
        ),
        "of",
        int(
            analysis[
                outcome
            ].notna().sum()
        )
    )

    display(
        tab
        .sort_values("q_FDR")
        .round(4)
    )


binary_results = pd.concat(
    binary_tables,
    ignore_index=True
)


binary_results.to_csv(
    OUT / "Table4_binary_logistic_regression_FDR.csv",
    index=False
)


# ============================================================
# 11. RANDOM FOREST
#     5-FOLD CROSS-VALIDATED PERFORMANCE
# ============================================================

rf_performance = []
rf_importance_tables = {}


# ------------------------------------------------------------
# Random Forest regression
# Fear and Emotional Effects
# ------------------------------------------------------------

def rf_regression_analysis(
    data,
    outcome,
    predictors
):

    d = data[
        [outcome] + predictors
    ].dropna().copy()

    X = d[
        predictors
    ].astype(float)

    y = d[
        outcome
    ].astype(float)


    model = RandomForestRegressor(
        n_estimators=500,
        min_samples_leaf=3,
        random_state=42,
        n_jobs=-1
    )


    cv = KFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )


    scores = cross_validate(
        model,
        X,
        y,
        cv=cv,
        scoring={
            "MAE":
                "neg_mean_absolute_error",

            "R2":
                "r2"
        },
        n_jobs=-1
    )


    perf = {
        "Outcome":
            outcome,

        "Model":
            "Random Forest regression",

        "N":
            len(d),

        "CV_MAE_mean":
            -scores[
                "test_MAE"
            ].mean(),

        "CV_MAE_SD":
            scores[
                "test_MAE"
            ].std(),

        "CV_R2_mean":
            scores[
                "test_R2"
            ].mean(),

        "CV_R2_SD":
            scores[
                "test_R2"
            ].std()
    }


    # Fit full dataset only for feature-ranking purposes
    model.fit(
        X,
        y
    )


    perm = permutation_importance(
        model,
        X,
        y,
        n_repeats=10,
        random_state=42,
        scoring="neg_mean_absolute_error",
        n_jobs=-1
    )


    imp = pd.DataFrame({
        "Outcome":
            outcome,

        "Predictor":
            predictors,

        "Permutation_importance":
            perm.importances_mean,

        "Importance_SD":
            perm.importances_std
    })


    imp = imp.sort_values(
        "Permutation_importance",
        ascending=False
    )


    return perf, imp


# ------------------------------------------------------------
# Random Forest classification
# Binary outcomes
# ------------------------------------------------------------

def rf_classification_analysis(
    data,
    outcome,
    predictors
):

    d = data[
        [outcome] + predictors
    ].dropna().copy()


    X = d[
        predictors
    ].astype(float)


    y = d[
        outcome
    ].astype(int)


    model = RandomForestClassifier(
        n_estimators=500,
        min_samples_leaf=3,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )


    cv = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )


    scores = cross_validate(
        model,
        X,
        y,
        cv=cv,
        scoring={
            "ROC_AUC":
                "roc_auc",

            "Balanced_accuracy":
                "balanced_accuracy"
        },
        n_jobs=-1
    )


    perf = {
        "Outcome":
            outcome,

        "Model":
            "Random Forest classification",

        "N":
            len(d),

        "Events":
            int(y.sum()),

        "CV_ROC_AUC_mean":
            scores[
                "test_ROC_AUC"
            ].mean(),

        "CV_ROC_AUC_SD":
            scores[
                "test_ROC_AUC"
            ].std(),

        "CV_balanced_accuracy_mean":
            scores[
                "test_Balanced_accuracy"
            ].mean(),

        "CV_balanced_accuracy_SD":
            scores[
                "test_Balanced_accuracy"
            ].std()
    }


    model.fit(
        X,
        y
    )


    perm = permutation_importance(
        model,
        X,
        y,
        n_repeats=10,
        random_state=42,
        scoring="roc_auc",
        n_jobs=-1
    )


    imp = pd.DataFrame({
        "Outcome":
            outcome,

        "Predictor":
            predictors,

        "Permutation_importance":
            perm.importances_mean,

        "Importance_SD":
            perm.importances_std
    })


    imp = imp.sort_values(
        "Permutation_importance",
        ascending=False
    )


    return perf, imp


# ------------------------------------------------------------
# Run RF regression
# ------------------------------------------------------------

for outcome in [
    "Fear",
    "Emotional effects"
]:

    perf, imp = (
        rf_regression_analysis(
            analysis,
            outcome,
            predictors
        )
    )

    rf_performance.append(
        perf
    )

    rf_importance_tables[
        outcome
    ] = imp


# ------------------------------------------------------------
# Run RF classification
# ------------------------------------------------------------

for outcome in [
    "Sleep avoidance",
    "Daily functioning",
    "Help-seeking"
]:

    perf, imp = (
        rf_classification_analysis(
            analysis,
            outcome,
            predictors
        )
    )

    rf_performance.append(
        perf
    )

    rf_importance_tables[
        outcome
    ] = imp


# Performance table
rf_perf = pd.DataFrame(
    rf_performance
)


print(
    "\nRANDOM FOREST CROSS-VALIDATION PERFORMANCE"
)

display(
    rf_perf.round(4)
)


rf_perf.to_csv(
    OUT /
    "Table5_Random_Forest_cross_validation_performance.csv",
    index=False
)


# All feature importance values
all_rf_imp = pd.concat(
    rf_importance_tables.values(),
    ignore_index=True
)


all_rf_imp.to_csv(
    OUT /
    "Random_Forest_permutation_importance_all_outcomes.csv",
    index=False
)


# ============================================================
# 12. RANDOM FOREST FEATURE IMPORTANCE FIGURES
# ============================================================

# ============================================================
# UPDATED RANDOM FOREST FEATURE IMPORTANCE FIGURES
# Same VIRIDIS color scheme as correlation matrix
# Larger publication-ready text
# ============================================================

import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import re

for outcome, imp in rf_importance_tables.items():

    # Keep top 10 predictors
    top = (
        imp
        .sort_values("Permutation_importance")
        .tail(10)
        .copy()
    )

    values = top["Permutation_importance"].values

    # --------------------------------------------------------
    # Match correlation matrix: VIRIDIS
    # Normalize colors according to importance values
    # --------------------------------------------------------

    vmin = values.min()
    vmax = values.max()

    # Protect against all values being identical
    if np.isclose(vmin, vmax):
        norm = mpl.colors.Normalize(
            vmin=vmin - 0.01,
            vmax=vmax + 0.01
        )
    else:
        norm = mpl.colors.Normalize(
            vmin=vmin,
            vmax=vmax
        )

    cmap = plt.cm.viridis

    bar_colors = cmap(
        norm(values)
    )

    # --------------------------------------------------------
    # Plot
    # --------------------------------------------------------

    fig, ax = plt.subplots(
        figsize=(11, 7)
    )

    bars = ax.barh(
        top["Predictor"],
        values,
        color=bar_colors
    )

    # Larger labels
    ax.set_xlabel(
        "Permutation importance",
        fontsize=17
    )

    ax.set_ylabel(
        "Predictor",
        fontsize=17
    )

    ax.tick_params(
        axis="x",
        labelsize=15
    )

    ax.tick_params(
        axis="y",
        labelsize=15
    )

    # Add the value at the end of each bar
    value_range = max(values.max() - values.min(), 0.01)

    for bar, value in zip(bars, values):

        ax.text(
            bar.get_width() + value_range * 0.015,
            bar.get_y() + bar.get_height() / 2,
            f"{value:.3f}",
            va="center",
            fontsize=13
        )

    # --------------------------------------------------------
    # Add matching viridis color scale
    # --------------------------------------------------------

    sm = mpl.cm.ScalarMappable(
        cmap=cmap,
        norm=norm
    )

    sm.set_array([])

    cbar = fig.colorbar(
        sm,
        ax=ax,
        fraction=0.035,
        pad=0.04
    )

    cbar.set_label(
        "Permutation importance",
        fontsize=15
    )

    cbar.ax.tick_params(
        labelsize=13
    )

    # Clean appearance
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    # No figure title inside plot — JEI caption will contain title
    plt.tight_layout()

    safe = re.sub(
        r"[^A-Za-z0-9]+",
        "_",
        outcome
    ).strip("_")

    plt.savefig(
        OUT / f"Figure_RF_feature_importance_{safe}.png",
        dpi=600,
        bbox_inches="tight"
    )

    plt.show()


# ============================================================
# SHAP INTERPRETATION FOR RANDOM FOREST MODELS
# Primary outcomes: Fear and Emotional Effects
# ============================================================

!pip -q install shap

import shap
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import re

shap_tables = {}

for outcome in [
    "Fear",
    "Emotional effects"
]:

    d = analysis[
        [outcome] + predictors
    ].dropna().copy()

    X = d[
        predictors
    ].astype(float)

    y = d[
        outcome
    ].astype(float)

    # Fit final RF model on full analytic dataset
    # Cross-validated performance should still be reported separately
    rf_model = RandomForestRegressor(
        n_estimators=500,
        min_samples_leaf=3,
        random_state=42,
        n_jobs=-1
    )

    rf_model.fit(X, y)

    # TreeExplainer is appropriate for Random Forest
    explainer = shap.TreeExplainer(
        rf_model
    )

    shap_values = explainer(
        X
    )

    # --------------------------------------------------------
    # 1. Mean absolute SHAP importance table
    # --------------------------------------------------------

    mean_abs_shap = np.abs(
        shap_values.values
    ).mean(axis=0)

    shap_importance = pd.DataFrame({
        "Outcome": outcome,
        "Predictor": predictors,
        "Mean_absolute_SHAP": mean_abs_shap
    }).sort_values(
        "Mean_absolute_SHAP",
        ascending=False
    )

    shap_tables[
        outcome
    ] = shap_importance

    display(
        shap_importance
        .round(4)
    )

    safe = re.sub(
        r"[^A-Za-z0-9]+",
        "_",
        outcome
    ).strip("_")

    shap_importance.to_csv(
        OUT /
        f"SHAP_importance_{safe}.csv",
        index=False
    )

    # --------------------------------------------------------
    # 2. SHAP BAR PLOT
    # Mean absolute SHAP values
    # --------------------------------------------------------

    shap.plots.bar(
        shap_values,
        max_display=10,
        show=False
    )

    plt.gcf().set_size_inches(
        9,
        6
    )

    plt.xticks(
        fontsize=14
    )

    plt.yticks(
        fontsize=14
    )

    plt.xlabel(
        "Mean absolute SHAP value",
        fontsize=15
    )

    plt.tight_layout()

    plt.savefig(
        OUT /
        f"Figure_SHAP_bar_{safe}.png",
        dpi=600,
        bbox_inches="tight"
    )

    plt.show()

    # --------------------------------------------------------
    # 3. SHAP BEESWARM PLOT
    # Shows magnitude AND direction of effects
    # --------------------------------------------------------

    shap.plots.beeswarm(
        shap_values,
        max_display=10,
        show=False
    )

    plt.gcf().set_size_inches(
        10,
        7
    )

    plt.xticks(
        fontsize=14
    )

    plt.yticks(
        fontsize=14
    )

    plt.xlabel(
        "SHAP value",
        fontsize=15
    )

    plt.tight_layout()

    plt.savefig(
        OUT /
        f"Figure_SHAP_beeswarm_{safe}.png",
        dpi=600,
        bbox_inches="tight"
    )

    plt.show()


# Combine SHAP importance tables
all_shap = pd.concat(
    shap_tables.values(),
    ignore_index=True
)

all_shap.to_csv(
    OUT /
    "SHAP_importance_all_primary_outcomes.csv",
    index=False
)

print(
    "SHAP analyses complete."
)


# ============================================================
# 13. AUTOMATIC RESULTS SUMMARY
# ============================================================

summary_lines = []

summary_lines.append(
    f"Final analytic sample: N={len(analysis)}."
)


summary_lines.append(
    "\nOrdinal logistic regression: "
    "statistically significant predictors (p<0.05):"
)


for outcome in [
    "Fear",
    "Emotional effects"
]:

    sub = ordinal_results[
        (
            ordinal_results[
                "Outcome"
            ] == outcome
        )
        &
        (
            ordinal_results[
                "p_value"
            ] < 0.05
        )
    ].sort_values(
        "p_value"
    )


    summary_lines.append(
        f"\n{outcome}:"
    )


    if len(sub) == 0:

        summary_lines.append(
            "  None at p<0.05."
        )

    else:

        for _, r in sub.iterrows():

            summary_lines.append(
                f"  {r['Predictor']}: "
                f"OR={r['OR']:.2f}, "
                f"95% CI "
                f"{r['CI95_low']:.2f}-"
                f"{r['CI95_high']:.2f}, "
                f"p={r['p_value']:.4g}"
            )


summary_lines.append(
    "\nBinary logistic regression: "
    "FDR-significant predictors (q<0.05):"
)


for outcome in [
    "Sleep avoidance",
    "Daily functioning",
    "Help-seeking"
]:

    sub = binary_results[
        (
            binary_results[
                "Outcome"
            ] == outcome
        )
        &
        (
            binary_results[
                "q_FDR"
            ] < 0.05
        )
    ].sort_values(
        "q_FDR"
    )


    summary_lines.append(
        f"\n{outcome}:"
    )


    if len(sub) == 0:

        summary_lines.append(
            "  None at q<0.05."
        )

    else:

        for _, r in sub.iterrows():

            summary_lines.append(
                f"  {r['Predictor']}: "
                f"OR={r['OR']:.2f}, "
                f"95% CI "
                f"{r['CI95_low']:.2f}-"
                f"{r['CI95_high']:.2f}, "
                f"q={r['q_FDR']:.4g}"
            )


summary_lines.append(
    "\nRandom Forest cross-validation:"
)


for _, r in rf_perf.iterrows():

    if (
        r["Model"] ==
        "Random Forest regression"
    ):

        summary_lines.append(
            f"  {r['Outcome']}: "
            f"CV MAE="
            f"{r['CV_MAE_mean']:.3f}, "
            f"CV R2="
            f"{r['CV_R2_mean']:.3f}"
        )

    else:

        summary_lines.append(
            f"  {r['Outcome']}: "
            f"CV ROC AUC="
            f"{r['CV_ROC_AUC_mean']:.3f}, "
            f"balanced accuracy="
            f"{r['CV_balanced_accuracy_mean']:.3f}"
        )


summary_text = "\n".join(
    summary_lines
)


print(
    "\n" +
    "=" * 75
)

print(
    "RESULTS SUMMARY FOR MANUSCRIPT REVIEW"
)

print(
    "=" * 75
)

print(
    summary_text
)


with open(
    OUT /
    "Results_summary_for_manuscript.txt",
    "w"
) as f:

    f.write(
        summary_text
    )


# ============================================================
# 14. SAVE CLEAN ANALYTIC DATASET
# ============================================================

analysis.to_csv(
    OUT /
    "Clean_analytic_dataset.csv",
    index=False
)


# ============================================================
# 15. ZIP ALL OUTPUTS
# ============================================================

zip_path = (
    "/content/"
    "Myna_SP_Final_Analysis_Outputs.zip"
)


with zipfile.ZipFile(
    zip_path,
    "w",
    zipfile.ZIP_DEFLATED
) as z:

    for file in OUT.iterdir():

        if file.is_file():

            z.write(
                file,
                arcname=file.name
            )


print(
    "\nDONE."
)

print(
    "Output folder:",
    OUT
)

print(
    "ZIP file:",
    zip_path
)


# ============================================================
# 16. DOWNLOAD ZIP
# ============================================================

try:

    from google.colab import files

    files.download(
        zip_path
    )

except Exception:

    pass